# Challenge 4: Evaluators, including span level

**EDD: Measure. Model upgrade framework, step 4 (build): choose the evaluators.** An evaluator turns one output into a score. You'll run code and LLM-as-judge evaluators locally, score individual
**spans** (a single tool call), and score whole multi-turn **threads**. Challenge 7 then checks that the judge can be trusted.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

import inspect

from eval_lab.agent import SYSTEM_PROMPT_V2, build_agent
from eval_lab.evaluators import (EVALUATOR_VERSION, answer_quality, exact_trajectory,
                                 judge_json, keyword_match, make_run_agent)
from utils.models import BASELINE_ID

## 1. Local evaluators on a dataset

An evaluator is a plain function returning `{"key", "score", "comment"}`. Three are defined in
`eval_lab/evaluators.py`: two in code, one LLM judge whose rationale becomes the comment. Read the judge:

In [ ]:
print(inspect.getsource(exact_trajectory))
print(inspect.getsource(answer_quality))
print(inspect.getsource(judge_json))

Run all three against the `test` split (12 examples). The experiment's metadata records the model and the evaluator
version, so a score is never separated from what produced it.

In [ ]:
results = client.evaluate(
    make_run_agent(build_agent(SYSTEM_PROMPT_V2)),
    data=client.list_examples(dataset_name=GOLDEN, splits=["test"]),
    evaluators=[exact_trajectory, keyword_match, answer_quality],
    experiment_prefix="eval-local",
    metadata={"model": BASELINE_ID, "evaluators": EVALUATOR_VERSION},
    max_concurrency=4,
)
EXPERIMENT = results.experiment_name
print("experiment:", EXPERIMENT)

## 2. Scores, rationale and failures

Scores live as feedback on each run. A judge can fail (bad JSON, throttling). `judge_json` never raises: it records
`score=None` and the failure reason, so a failed judgement is visible instead of silently averaged away.

In [ ]:
import time
time.sleep(5)  # feedback is written asynchronously
exp_runs = list(client.list_runs(project_name=EXPERIMENT, is_root=True, select=["id"]))
feedback = list(client.list_feedback(run_ids=[r.id for r in exp_runs], feedback_key=["answer_quality"]))
for fb in feedback:
    print(fb.score, "|", (fb.comment or "")[:100])
print("judge failures:", sum(1 for fb in feedback if fb.score is None))

## 3. Span-level evaluators

Trace and thread evaluators answer "was the final answer good?". They can't tell you *which step* went wrong.
For routing and tool questions (was the right tool called, did the call succeed, were the arguments valid) you
score the **span**: fetch tool runs with `run_type="tool"` and attach feedback to each one with
`client.create_feedback(run_id=<span id>, session_id=<span.session_id>, ...)`.

In [ ]:
tool_runs = list(client.list_runs(project_name=PROD, run_type="tool"))
from collections import Counter
print(len(tool_runs), "tool spans:", dict(Counter(r.name for r in tool_runs)))

Write `evaluate_tool_span(run)`. It returns `{"key": "tool_span_ok", "score": 0 or 1, "comment": ...}`:
a span fails if the tool errored, or if `order_id` isn't of the form `A` plus four digits, or if `search_faq` got an
empty query. A reference solution is in the next cell.

In [ ]:
# Reference solution: try writing this yourself first.
import re

def evaluate_tool_span(run):
    args = lab_sdk.tool_args(run)
    if run.error:
        return {"key": "tool_span_ok", "score": 0, "comment": f"tool errored: {str(run.error).split('Traceback')[0][:80]}"}
    if run.name == "search_faq":
        return {"key": "tool_span_ok", "score": int(bool(args.get("query")))}
    order_id = args.get("order_id")
    ok = bool(re.fullmatch(r"A\d{4}", str(order_id)))
    return {"key": "tool_span_ok", "score": int(ok), "comment": f"order_id={order_id!r}"}

In [ ]:
assert evaluate_tool_span(tool_runs[0])["key"] == "tool_span_ok", "evaluate_tool_span must return key 'tool_span_ok'"

bad = []
for run in tool_runs:
    result = evaluate_tool_span(run)
    client.create_feedback(run_id=run.id, session_id=run.session_id, **result,
                           source_info={"evaluator": "tool_span_ok@v1"})
    if result["score"] == 0:
        bad.append((run.name, result.get("comment")))
client.flush()

from lab_state import mark_done
mark_done("/tmp/.sdk-ch4-span-ok")
print(f"scored {len(tool_runs)} tool spans; {len(bad)} failed:")
for name, comment in bad:
    print("  ", name, "-", comment)

## 4. Multi-turn: score a whole thread

Runs that share a `thread_id` in metadata form a conversation. A thread evaluator reads the transcript and scores
the conversation, not one turn. The score is attached to the thread's last run.

In [ ]:
threads = {}
for r in lab_sdk.roots(client):
    tid = lab_sdk.run_meta(r).get("thread_id")
    if tid:
        threads.setdefault(tid, []).append(r)

THREAD_JUDGE = (
    "Below is a customer-support conversation.\n{transcript}\n"
    "Did the agent resolve the customer's requests across the whole conversation, without contradicting itself "
    "or granting something it should not? Reply with only JSON: "
    '{{"correct": 0 or 1, "reason": "one sentence"}}'
)

for tid, turns in sorted(threads.items()):
    turns.sort(key=lambda r: r.start_time)
    transcript = "\n".join(
        f"Customer: {lab_sdk.redact(str(t.inputs.get('query')))}\nAgent: {(t.outputs or {}).get('answer')}" for t in turns)
    verdict = judge_json(THREAD_JUDGE.format(transcript=transcript))
    client.create_feedback(run_id=turns[-1].id, session_id=turns[-1].session_id, key="thread_resolved",
                           score=verdict.get("correct"), comment=verdict.get("reason"), source_info={"evaluator": "thread_resolved@v1"})
    print(tid, len(turns), "turns ->", verdict.get("correct"), "|", verdict.get("reason"))
client.flush()

## 5. Optional: evaluators configured in LangSmith

Some evaluators are configured in the LangSmith UI instead of code, for example an LLM evaluator attached to a dataset.
Those run **remotely**, in LangSmith, whenever a new experiment is created. To try it:

1. Open the dataset `sdk-golden-...` in LangSmith, then **Evaluators**, then add an LLM-as-judge evaluator.
2. Re-run the `client.evaluate(...)` cell in step 1.
3. Run the cell below. It lists any score keys that no local evaluator produced; those came from the remote evaluator.

This step depends on your LangSmith plan.

In [ ]:
LOCAL = {"exact_trajectory", "keyword_match", "answer_quality"}
latest = next(iter(sorted(client.list_projects(reference_dataset_id=client.read_dataset(dataset_name=GOLDEN).id),
                          key=lambda p: p.start_time, reverse=True)))
ids = [r.id for r in client.list_runs(project_name=latest.name, is_root=True, select=["id"])]
remote = sorted({fb.key for fb in client.list_feedback(run_ids=ids)} - LOCAL)
print("latest experiment:", latest.name)
print("remote evaluator keys:", remote or "none yet (configure an evaluator in the UI, then re-run the evaluate cell)")

## 6. See it in LangSmith

In [ ]:
lab_links.project_link(PROD, hint="filter runs by Run type = tool, and open the tool_span_ok and thread_resolved feedback")
lab_links.dataset_link(GOLDEN, hint="Experiments tab: the eval-local experiment with its scores and judge comments")